# Data Audit


## Purpose
Verify fetched data: row counts per season per source, schema mismatches, team name coverage.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
sys_path = str(Path(".").parent / "src")
import sys; sys.path.insert(0, sys_path)
from nba_predictor.config import cfg
print("Config loaded:", cfg.seasons)


## Team Stats Coverage

In [ ]:
team_path = cfg.project_root / "data/raw/bball_ref/team_stats/team_advanced_all.parquet"
if team_path.exists():
    df = pd.read_parquet(team_path)
    print(f"Team stats: {len(df)} rows, seasons {df.season.min()}–{df.season.max()}")
    print(df.groupby("season").size().describe())
else:
    print("No data yet — run make fetch")


# CLAUDE: are these the expected numbers from this check?

## Playoff Series Coverage

In [ ]:
series_path = cfg.project_root / "data/raw/bball_ref/playoff_series/playoff_series_all.parquet"
if series_path.exists():
    df = pd.read_parquet(series_path)
    print(f"Playoff series: {len(df)} rows")
    print(df.groupby("season").size())
else:
    print("No data yet — run make fetch")


## Team Name Standardization Check

In [ ]:
# Verify all team names in data map to canonical abbreviations
# Flag any unknowns that need to be added to conf/config.yaml
if series_path.exists():
    df = pd.read_parquet(series_path)
    all_teams = set(df["team_a"].tolist() + df["team_b"].tolist())
    print(f"Unique teams: {len(all_teams)}")
    for t in sorted(all_teams):
        print(t)


---
## Data Quality & Gaps Analysis

This section documents known data coverage gaps, null patterns, and anomalies across the full 1984–2026 dataset. Understanding which features are available for which eras is critical for correct interpretation of model predictions.

### Feature Coverage by Era

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Load processed series dataset (model input)
series_path = cfg.project_root / 'data/processed/series_dataset.parquet'
if not series_path.exists():
    print('Run make process first')
else:
    df = pd.read_parquet(series_path)
    print(f'Series dataset: {len(df)} rows x {df.shape[1]} cols')
    print(f'Training range: {df.season.min()}–{df.season.max()}')
    print()
    # Feature null rates across all rows
    null_pct = (df.isnull().sum() / len(df) * 100).sort_values(ascending=False)
    print('Features with any nulls:')
    print(null_pct[null_pct > 0].to_string())
    print()
    print('Total features with zero nulls:', (null_pct == 0).sum())

In [ ]:
# Feature availability by era
if series_path.exists():
    eras = [
        ('Pre-1996 (no API)', df.season < 1996),
        ('1996-2002 (API, pre best-of-7)', (df.season >= 1996) & (df.season < 2003)),
        ('2003-2014 (model training, early)', (df.season >= 2003) & (df.season <= 2014)),
        ('2015-2026 (analytics era)', df.season >= 2015),
    ]
    key_features = [
        'delta_NRtg', 'delta_VORP', 'delta_L10_NRtg', 'H2H_win_pct',
        'H2H_NRtg_avg', 'higher_has_injury_data', 'seed_diff', 'era_analytics'
    ]
    rows = []
    for era_name, mask in eras:
        era_df = df[mask]
        row = {'Era': era_name, 'N series': len(era_df)}
        for feat in key_features:
            if feat in era_df.columns:
                null_rate = era_df[feat].isnull().mean() * 100
                row[feat] = f'{100-null_rate:.0f}% avail'
            else:
                row[feat] = 'NOT IN DATA'
        rows.append(row)
    coverage = pd.DataFrame(rows).set_index('Era')
    print('Feature availability by era:')
    print(coverage.to_string())

### Injury Data Status

In [ ]:
# Document injury data situation explicitly
if series_path.exists():
    inj_cols = [c for c in df.columns if 'injury' in c.lower() or 'has_injury' in c.lower() or 'Star_injured' in c]
    print('Injury-related columns in series_dataset:')
    for col in inj_cols:
        unique_vals = df[col].dropna().unique()
        null_count = df[col].isnull().sum()
        print(f'  {col}: unique values={sorted(unique_vals[:5])}, nulls={null_count}')
    print()
    if 'higher_has_injury_data' in df.columns:
        print(f'has_injury_data = 1 in {(df["higher_has_injury_data"]==1).sum()} rows (out of {len(df)})')
        print(f'has_injury_data = 0 in {(df["higher_has_injury_data"]==0).sum()} rows')
        print()
        print('NOTE: has_injury_data is 0 for ALL training rows.')
        print('Injury-related features (Star_injured, Roster_VORP_available_pct, etc.)')
        print('are computed from roster/minutes data as a proxy, not from actual injury reports.')
        print('This means the model was trained WITHOUT real injury signal.')
        print('The 2026 predictions also lack real injury data — this is consistent but a limitation.')

### H2H Feature Coverage

In [ ]:
# H2H features: available from 1996+
if series_path.exists():
    h2h_cols = ['H2H_win_pct', 'H2H_NRtg_avg', 'H2H_games_played']
    for col in h2h_cols:
        if col in df.columns:
            null_count = df[col].isnull().sum()
            pre96_null = df[df.season < 1996][col].isnull().sum()
            post96_null = df[df.season >= 1996][col].isnull().sum()
            print(f'{col}: total nulls={null_count}, pre-1996={pre96_null}/{len(df[df.season<1996])}, post-1996={post96_null}/{len(df[df.season>=1996])}')
    print()
    # Check 2026 bracket input
    bracket_path = cfg.project_root / 'data/predictions/2026/bracket_input.csv'
    if bracket_path.exists():
        b = pd.read_csv(bracket_path)
        print('2026 bracket_input H2H status:')
        for col in h2h_cols:
            if col in b.columns:
                null_count = b[col].isnull().sum()
                print(f'  {col}: {null_count}/{len(b)} rows are null/blank')
        print()
        print('NOTE: H2H is blank for 2026 because the 2025-26 game logs were not')
        print('fetched when build_bracket_input.py was run. The model fills with 0.')
        print('H2H was populated in training data for all 1996+ seasons.')
        print('This is a known gap — see Phase 2 fix plan.')

### Series Length Distribution by Era

In [ ]:
# Has series length distribution shifted across eras?
if series_path.exists():
    eras_len = [
        (2003, 2009, 'Early analytics (2003-2009)'),
        (2010, 2014, 'Mid analytics (2010-2014)'),
        (2015, 2019, 'Late analytics (2015-2019)'),
        (2020, 2026, 'Modern era (2020-2026)'),
    ]
    print('Series length distribution by era (% of series):')
    print(f'{"Era":<30} {"4 games":>9} {"5 games":>9} {"6 games":>9} {"7 games":>9} {"Mean":>8}')
    for start, end, label in eras_len:
        mask = (df.season >= start) & (df.season <= end)
        era_df = df[mask]
        if len(era_df) == 0:
            continue
        counts = era_df['series_length'].value_counts(normalize=True).sort_index() * 100
        mean_len = era_df['series_length'].mean()
        print(f'{label:<30}', end='')
        for g in [4, 5, 6, 7]:
            print(f' {counts.get(g, 0):>8.1f}%', end='')
        print(f' {mean_len:>7.2f}')
    print()
    print('Overall:', df['series_length'].value_counts(normalize=True).sort_index().apply(lambda x: f'{x*100:.1f}%').to_dict())

### Aberrant Seasons

In [ ]:
# Flag seasons with known data quality or structural differences
print('ABERRANT SEASONS:')
print()
print('2012 — NBA Lockout')
print('  Only 66 games played (vs 82 normal).')
print('  Teams have fewer games for momentum features (L10_NRtg, etc.)')
print('  Win% stats may be less stable due to compressed schedule.')
print('  Flagged with season_flag=1 in model features.')
if series_path.exists():
    s2012 = df[df.season == 2012]
    print(f'  Series in 2012: {len(s2012)} — all modeled but treated with caution')
print()
print('2020 — NBA Bubble (Orlando)')
print('  No home court advantage (all games at neutral Disney site).')
print('  The home_court_advantage feature is always 1 in the model (not zeroed for 2020).')
print('  home_court_boost=2.5 in Monte Carlo is incorrect for this season.')
print('  Flagged with season_flag=1 in model features.')
if series_path.exists():
    s2020 = df[df.season == 2020]
    print(f'  Series in 2020: {len(s2020)} — HCA was ~0 historically, model may underestimate upsets')
print()
print('2026 — Current prediction year')
print('  Game logs available through regular season (April 17 cutoff).')
print('  H2H features currently blank (game logs not fetched).')
print('  Injury data: all has_injury_data=0 (same as training — consistent but limited).')

### Delta Feature Distribution — 2026 vs. Training

In [ ]:
# Are 2026 bracket inputs within the training data distribution?
if series_path.exists():
    bracket_path = cfg.project_root / 'data/predictions/2026/bracket_input.csv'
    if bracket_path.exists():
        b = pd.read_csv(bracket_path)
        key_delta_features = ['delta_NRtg', 'delta_BPM', 'delta_VORP', 'delta_L10_NRtg', 'delta_Star_BPM', 'seed_diff']
        print('Feature distribution comparison (2026 bracket vs training data):')
        print(f'{"Feature":<25} {"Train min":>9} {"Train max":>9} {"Train mean":>11} | {"2026 range":>18}')
        print('-' * 80)
        for feat in key_delta_features:
            if feat in df.columns and feat in b.columns:
                t_min, t_max, t_mean = df[feat].min(), df[feat].max(), df[feat].mean()
                b_min, b_max = b[feat].min(), b[feat].max()
                out_of_range = b[(b[feat] < t_min) | (b[feat] > t_max)]
                flag = ' ⚠️ OOD' if len(out_of_range) > 0 else ''
                print(f'{feat:<25} {t_min:>9.2f} {t_max:>9.2f} {t_mean:>11.2f} | {b_min:>8.2f} to {b_max:>8.2f}{flag}')
        print()
        print('⚠️ OOD = Out of Distribution: 2026 value outside training data range')

### Summary: Known Data Gaps and Limitations

| Gap | Affects | Status |
|-----|---------|--------|
| Injury reports not fetched | `has_injury_data=0` for all rows | Consistent (train+predict), but no real injury signal |
| H2H blank for 2026 | `H2H_win_pct`, `H2H_NRtg_avg` null in bracket_input | Mismatch with training data (1996+ has H2H); Phase 2 fix |
| 2020 bubble HCA | `home_court_advantage=1` even for 2020 | Model slightly miscalibrated for 2020 series |
| L10 momentum: pre-1996 | `delta_L10_NRtg` null for pre-1996 | Filled with 0; no signal from early eras |
| LightGBM length model | Trained on 43 features, dataset has 49 | 6 new features unused; Phase 2 fix |
| Ensemble model overfit | 300-estimator trees on 345 samples | 0% of 2026 predictions used ensemble (all used MC fallback); Phase 2 fix |
| Competitive_balance_index | Always null | Not used in model; metadata only |